In [1]:
import os

REPO_URL = "https://github.com/sinh2206/O_D.git"
REPO_DIR = "O_D"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch --all --prune
    !git checkout -B $(git symbolic-ref --short refs/remotes/origin/HEAD | sed 's@^origin/@@')
    !git reset --hard origin/$(git symbolic-ref --short refs/remotes/origin/HEAD | sed 's@^origin/@@')
    %cd ..

%cd {REPO_DIR}
!git log -1 --oneline


Cloning into 'O_D'...
remote: Enumerating objects: 9548, done.
remote: Counting objects: 100% (517/517), done.
remote: Compressing objects: 100% (411/411), done.
remote: Total 9548 (delta 248), reused 369 (delta 104), pack-reused 9031 (from 2)
Receiving objects: 100% (9548/9548), 1009.98 MiB | 59.59 MiB/s, done.
Resolving deltas: 100% (249/249), done.
Updating files: 100% (9023/9023), done.
/kaggle/working/O_D
dc3913c (HEAD -> main, origin/main, origin/HEAD) 2.1


In [2]:
!pwd
!ls


/kaggle/working/O_D
img_error.py  models	       predict.py  README.md	     results   utils
LICENSE       obj-detec.ipynb  public	   requirements.txt  train.py


In [3]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.5 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requ

In [4]:
!python train.py \
  --train_data ./public/annotations/train.json \
  --val_data ./public/annotations/val.json \
  --image_dir ./public/train/images \
  --val_image_dir ./public/val/images \
  --checkpoint_dir ./models/

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100%|███████████████████████████████████████| 83.3M/83.3M [00:00<00:00, 192MB/s]
Device: cuda, AMP: True
Train samples: 7500, Val samples: 1500, Classes: ['person', 'car', 'dog', 'cat', 'chair']
Balanced sampling: True
Class weights enabled: True
Num workers: requested=2, resolved=2, max_safe=4
Pin memory: True
Class weights: [0.45, 0.9275, 1.1709, 1.4433, 1.249]
Epoch 001/030 | train_loss=1.6900 (cls=0.5575, reg=0.6588, ctr=0.6750) | val_loss=1.3656
Saved best checkpoint: models/best.pth (val_loss=1.3656)
Epoch 002/030 | train_loss=1.3113 (cls=0.4092, reg=0.5007, ctr=0.6627) | val_loss=1.1788
Saved best checkpoint: models/best.pth (val_loss=1.1788)
Epoch 003/030 | train_loss=1.1989 (cls=0.3858, reg=0.4335, ctr=0.6551) | val_loss=1.1311
Saved best checkpoint: models/best.pth (val_loss=1.1311)
Epoch 004/030 | train_loss=1.0830 (cls=0.3357, reg=0.3908, ctr=

In [5]:
!python predict.py \
  --image_dir ./public/val/images \
  --output val_predictions.json \
  --model_path ./models/best.pth

Device: cuda
Checkpoint meta: epoch=30, best_val_loss=0.8082473823364745, img_size=640
Predicted images: 1500
Saved JSON: val_predictions.json
TTA fallback: False (min_votes=2)
Hardcase items: 50
Hardcase summary: results/hardcase_summary.json


In [6]:
!python public/tools/evaluate_predictions.py \
  --ground_truth public/annotations/val.json \
  --predictions val_predictions.json \
  --output val_score.json

{
  "mAP@0.5": 0.725082,
  "performance_points": 20,
  "iou_threshold": 0.5,
  "num_ground_truth_boxes": 2021,
  "num_predictions": 3242,
  "micro_precision": 0.492906,
  "micro_recall": 0.790698,
  "per_class": {
    "person": {
      "ap": 0.778122,
      "num_ground_truth": 1074,
      "num_predictions": 1263,
      "true_positives": 864,
      "false_positives": 399,
      "recall": 0.804469,
      "precision": 0.684086
    },
    "car": {
      "ap": 0.700472,
      "num_ground_truth": 283,
      "num_predictions": 478,
      "true_positives": 217,
      "false_positives": 261,
      "recall": 0.766784,
      "precision": 0.453975
    },
    "dog": {
      "ap": 0.755597,
      "num_ground_truth": 206,
      "num_predictions": 385,
      "true_positives": 170,
      "false_positives": 215,
      "recall": 0.825243,
      "precision": 0.441558
    },
    "cat": {
      "ap": 0.835984,
      "num_ground_truth": 176,
      "num_predictions": 269,
      "true_positives": 154,
      "f

In [7]:
!python img_error.py

Rendered hardcase images: 50
Source summary: results/hardcase_summary.json
Predictions: val_predictions.json
Output dir: results


In [8]:
# Zip project excluding public
import os
from pathlib import Path

src = Path('/kaggle/working/O_D')
out_zip = Path('/kaggle/working/train.zip')

if not src.exists():
    raise FileNotFoundError(f'Not found: {src}')

cmd = f"cd {src} && zip -r {out_zip} . -x 'public/*' 'public/**'"
print(cmd)
ret = os.system(cmd)
if ret != 0:
    raise RuntimeError(f'zip failed with code {ret}')
print(f'Created: {out_zip}')


cd /kaggle/working/O_D && zip -r /kaggle/working/train.zip . -x 'public/*' 'public/**'
  adding: obj-detec.ipynb (deflated 80%)
  adding: .gitignore (deflated 10%)
  adding: val_predictions.json (deflated 90%)
  adding: models/ (stored 0%)
  adding: models/last.pth (deflated 8%)
  adding: models/README.md (stored 0%)
  adding: models/best.pth (deflated 8%)
  adding: val_score.json (deflated 71%)
  adding: requirements.txt (deflated 13%)
  adding: results/ (stored 0%)
  adding: results/hardcase_025_img_0d6d2588c38c.jpg (deflated 0%)
  adding: results/hardcase_018_img_5b35abeb2218.jpg (deflated 1%)
  adding: results/hardcase_044_img_a423367e9297.jpg (deflated 1%)
  adding: results/hardcase_033_img_046d0d26cad5.jpg (deflated 3%)
  adding: results/hardcase_002_img_4f10a2144ebe.jpg (deflated 0%)
  adding: results/hardcase_028_img_66bc8a30fa85.jpg (deflated 0%)
  adding: results/hardcase_007_img_95f0e0c36488.jpg (deflated 1%)
  adding: results/hardcase_017_img_51e582e7dc22.jpg (deflated 1%)
